# Ultrasound — primary model training

Resumable training of U-Net, Attention U-Net, and Swin-Tiny-U-Net on BUS-BRA, with validation-based model and threshold selection.


In [ ]:
!pip -q install scipy tqdm

In [ ]:
# Change only MODEL_NAME and SEED between training sessions.

MODEL_NAME = "unet"
SEED = 42

# The prepared dataset must contain:
DATA_ROOT_OVERRIDE = ""

OUTPUT_ROOT_OVERRIDE = ""

RESUME_CHECKPOINT_OVERRIDE = ""

IMAGE_SIZE = 256
NUM_WORKERS = 2
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 18
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

BATCH_SIZE_BY_MODEL = {
    "unet": 16,
    "attention_unet": 12,
    "swin_tiny_unet": 6,
}

BASE_LR_BY_MODEL = {
    "unet": 2e-4,
    "attention_unet": 2e-4,
    "swin_tiny_unet": 2e-4,
}
SWIN_ENCODER_LR = 2e-5

# Select the threshold using validation data only.
THRESHOLD_GRID = [
    round(value / 100, 2)
    for value in range(5, 100, 5)
]

ALLOW_EXTERNAL_ROWS = False

CREATE_LIGHT_ZIP = True

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import binary_erosion, distance_transform_edt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Kaggle must be detected first. Some Kaggle images can import google.colab,
IN_KAGGLE = (
    Path("/kaggle/input").exists()
    and Path("/kaggle/working").exists()
)

IN_COLAB = False
drive = None

if not IN_KAGGLE:
    try:
        from google.colab import drive
        IN_COLAB = True
    except Exception:
        IN_COLAB = False
        drive = None

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
    except Exception as error:
        print("Google Drive mount warning:", error)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
AMP_ENABLED = DEVICE.type == "cuda"

RUNTIME_NAME = (
    "kaggle"
    if IN_KAGGLE
    else (
        "colab"
        if IN_COLAB
        else "local"
    )
)

print(
    "Runtime environment:",
    {
        "runtime": RUNTIME_NAME,
        "kaggle": IN_KAGGLE,
        "colab": IN_COLAB,
        "device": str(DEVICE),
        "torch": torch.__version__,
    },
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

In [ ]:
# Robust dataset and output discovery
EXPECTED_BUSBRA_CROPS = 1875
EXPECTED_BREAST_EXTERNAL_CROPS = 252
EXPECTED_TOTAL_CROPS = (
    EXPECTED_BUSBRA_CROPS
    + EXPECTED_BREAST_EXTERNAL_CROPS
)

AUTO_EXTRACT_ROOT = Path(
    "/kaggle/working/TRACKC_DATA_AUTO_EXTRACTED"
    if IN_KAGGLE
    else "/content/TRACKC_DATA_AUTO_EXTRACTED"
)

def _safe_extract_zip(
    zip_path: Path,
    destination: Path,
) -> Path:
    marker = destination / ".extraction_complete"

    if marker.exists():
        return destination

    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(zip_path) as archive:
        destination_resolved = destination.resolve()

        for member in archive.infolist():
            target = (
                destination / member.filename
            ).resolve()

            if (
                target != destination_resolved
                and destination_resolved
                not in target.parents
            ):
                raise RuntimeError(
                    f"Unsafe ZIP member: {member.filename}"
                )

        archive.extractall(destination)

    marker.write_text(
        str(zip_path),
        encoding="utf-8",
    )

    print(
        "Extracted full dataset archive:",
        zip_path,
        "->",
        destination,
    )
    return destination

def _zip_contains_full_dataset(
    zip_path: Path,
) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = [
                name.replace("\\", "/")
                for name in archive.namelist()
            ]

        has_manifest = any(
            name.endswith(
                "roi_us_manifest.csv"
            )
            for name in names
        )
        npz_count = sum(
            name.lower().endswith(".npz")
            for name in names
        )

        return (
            has_manifest
            and npz_count >= EXPECTED_BUSBRA_CROPS
        )

    except Exception:
        return False

def _search_base_directories() -> List[Path]:
    bases = []

    if IN_KAGGLE:
        bases.extend(
            [
                Path("/kaggle/input"),
                AUTO_EXTRACT_ROOT,
            ]
        )

    if IN_COLAB:
        bases.extend(
            [
                Path(
                    "/content/drive/MyDrive/"
                    "TRACKC_ULTRASOUND"
                ),
                Path("/content"),
                AUTO_EXTRACT_ROOT,
            ]
        )

    bases.extend(
        [
            Path.cwd(),
            Path("/mnt/data"),
        ]
    )

    unique = []
    seen = set()

    for base in bases:
        try:
            key = str(base.resolve())
        except Exception:
            key = str(base)

        if key not in seen and base.exists():
            seen.add(key)
            unique.append(base)

    return unique

def _find_manifest_candidates() -> List[Path]:
    candidates = []

    for base in _search_base_directories():
        try:
            candidates.extend(
                base.rglob(
                    "roi_us_manifest.csv"
                )
            )
        except Exception:
            pass

    unique = []
    seen = set()

    for path in candidates:
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)

        if key not in seen:
            seen.add(key)
            unique.append(path)

    return unique

def _count_npz_under(
    root: Path,
    limit: Optional[int] = None,
) -> int:
    count = 0

    try:
        for _ in root.rglob("*.npz"):
            count += 1
            if limit is not None and count >= limit:
                break
    except Exception:
        pass

    return count

def _candidate_root_from_manifest(
    manifest_path: Path,
) -> Path:
    return manifest_path.parent

def _score_dataset_root(
    root: Path,
) -> Tuple[int, int, int]:
    npz_count = _count_npz_under(
        root,
        limit=EXPECTED_TOTAL_CROPS + 1,
    )

    exact_name_bonus = int(
        root.name == "ROI_US_Crops_256_v1"
    )
    full_name_bonus = int(
        "FULL" in root.name.upper()
    )
    audit_penalty = int(
        "AUDIT_LIGHT" in str(root).upper()
    )

    score = (
        npz_count * 100
        + exact_name_bonus * 10
        + full_name_bonus * 5
        - audit_penalty * 100000
    )

    return score, npz_count, exact_name_bonus

def _auto_extract_full_dataset_zip() -> bool:
    archive_candidates = []

    for base in _search_base_directories():
        try:
            archive_candidates.extend(
                base.rglob("*.zip")
            )
        except Exception:
            pass

    for archive_path in archive_candidates:
        if _zip_contains_full_dataset(
            archive_path
        ):
            destination = (
                AUTO_EXTRACT_ROOT
                / re.sub(
                    r"[^A-Za-z0-9_.-]+",
                    "_",
                    archive_path.stem,
                )
            )
            _safe_extract_zip(
                archive_path,
                destination,
            )
            return True

    return False

def discover_data_root() -> Path:
    if DATA_ROOT_OVERRIDE:
        root = Path(
            DATA_ROOT_OVERRIDE
        ).expanduser()

        if not root.exists():
            raise FileNotFoundError(
                "DATA_ROOT_OVERRIDE does not exist: "
                f"{root}"
            )

        manifest_path = (
            root / "roi_us_manifest.csv"
        )
        npz_count = _count_npz_under(
            root,
            limit=EXPECTED_TOTAL_CROPS + 1,
        )

        if not manifest_path.exists():
            raise FileNotFoundError(
                "DATA_ROOT_OVERRIDE does not contain "
                f"roi_us_manifest.csv: {root}"
            )

        if npz_count < EXPECTED_BUSBRA_CROPS:
            raise FileNotFoundError(
                "DATA_ROOT_OVERRIDE contains the manifest "
                f"but only {npz_count} NPZ files. "
                "Upload or mount the full "
                "ROI_US_Crops_256_v1 dataset, not the "
                "AUDIT_LIGHT package."
            )

        return root

    manifests = _find_manifest_candidates()

    scored = []
    for manifest_path in manifests:
        root = _candidate_root_from_manifest(
            manifest_path
        )
        score, npz_count, exact_bonus = (
            _score_dataset_root(root)
        )
        scored.append(
            {
                "root": root,
                "manifest": manifest_path,
                "score": score,
                "npz_count": npz_count,
            }
        )

    valid = [
        item
        for item in scored
        if item["npz_count"]
        >= EXPECTED_BUSBRA_CROPS
    ]

    if not valid:
        extracted = _auto_extract_full_dataset_zip()

        if extracted:
            manifests = _find_manifest_candidates()
            scored = []

            for manifest_path in manifests:
                root = _candidate_root_from_manifest(
                    manifest_path
                )
                score, npz_count, exact_bonus = (
                    _score_dataset_root(root)
                )
                scored.append(
                    {
                        "root": root,
                        "manifest": manifest_path,
                        "score": score,
                        "npz_count": npz_count,
                    }
                )

            valid = [
                item
                for item in scored
                if item["npz_count"]
                >= EXPECTED_BUSBRA_CROPS
            ]

    if not valid:
        diagnostic = pd.DataFrame(scored)

        if len(diagnostic):
            print(
                "Dataset candidates found:"
            )
            display(
                diagnostic[
                    [
                        "root",
                        "npz_count",
                        "manifest",
                    ]
                ]
            )

        raise FileNotFoundError(
            "The full Track C dataset was not found. "
            "The notebook found a manifest but could not "
            f"find at least {EXPECTED_BUSBRA_CROPS} NPZ files. "
            "The light audit ZIP is insufficient for training. "
            "Add ROI_US_Crops_256_v1_FULL.zip or the extracted "
            "ROI_US_Crops_256_v1 folder as a Kaggle input."
        )

    selected = sorted(
        valid,
        key=lambda item: item["score"],
        reverse=True,
    )[0]

    print(
        "Selected full dataset root:",
        selected["root"],
    )
    print(
        "NPZ files detected under selected root:",
        selected["npz_count"],
    )

    return Path(
        selected["root"]
    )

def discover_output_root() -> Path:
    if OUTPUT_ROOT_OVERRIDE:
        root = Path(
            OUTPUT_ROOT_OVERRIDE
        ).expanduser()

    elif IN_KAGGLE:
        root = Path(
            "/kaggle/working/"
            "TRACKC_PHASE2_RESULTS"
        )

    elif IN_COLAB:
        root = Path(
            "/content/drive/MyDrive/"
            "TRACKC_ULTRASOUND/"
            "TRACKC_PHASE2_RESULTS"
        )

    else:
        root = Path(
            "./TRACKC_PHASE2_RESULTS"
        )

    root.mkdir(
        parents=True,
        exist_ok=True,
    )
    return root

DATA_ROOT = discover_data_root()
OUTPUT_ROOT = discover_output_root()
RUN_ID = f"{MODEL_NAME}_seed{SEED}"
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    DATA_ROOT
    / "roi_us_manifest.csv"
)
BEST_PATH = RUN_DIR / "best.pt"
LAST_PATH = RUN_DIR / "last.pt"
HISTORY_PATH = RUN_DIR / "history.csv"
CONFIG_PATH = RUN_DIR / "config.json"

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("RUN_DIR:", RUN_DIR)

In [ ]:
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "dataset": str,
        "split": str,
        "patient_id": str,
        "case_id": str,
        "sample_id": str,
        "global_patient_id": str,
        "npz_path": str,
    },
)

manifest["dataset"] = (
    manifest["dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
)
manifest["split"] = (
    manifest["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)
manifest["patient_id"] = (
    manifest["patient_id"]
    .astype(str)
    .str.strip()
)
manifest["sample_id"] = (
    manifest["sample_id"]
    .astype(str)
    .str.strip()
)

if "global_patient_id" not in manifest.columns:
    manifest["global_patient_id"] = (
        manifest["dataset"]
        + "::"
        + manifest["patient_id"]
    )

required_columns = {
    "sample_id",
    "dataset",
    "split",
    "patient_id",
    "global_patient_id",
    "case_id",
    "npz_path",
}

missing_columns = (
    required_columns
    - set(manifest.columns)
)

if missing_columns:
    raise RuntimeError(
        "Manifest missing required columns: "
        f"{sorted(missing_columns)}"
    )

if not ALLOW_EXTERNAL_ROWS:
    external_in_training = manifest[
        (manifest["dataset"] == "breast")
        & (manifest["split"] != "external")
    ]

    if len(external_in_training):
        raise RuntimeError(
            "BrEaST contains non-external rows."
        )

development = manifest[
    (manifest["dataset"] == "bus_bra")
    & manifest["split"].isin(
        [
            "train",
            "validation",
            "test",
        ]
    )
].copy()

external_rows = manifest[
    (manifest["dataset"] == "breast")
    & (manifest["split"] == "external")
].copy()

if development.empty:
    raise RuntimeError(
        "BUS-BRA development rows are missing."
    )

if external_rows.empty:
    raise RuntimeError(
        "BrEaST external rows are missing from the manifest."
    )

if len(development) != EXPECTED_BUSBRA_CROPS:
    raise RuntimeError(
        "Unexpected BUS-BRA crop count: "
        f"{len(development)} observed versus "
        f"{EXPECTED_BUSBRA_CROPS} expected."
    )

if len(external_rows) != EXPECTED_BREAST_EXTERNAL_CROPS:
    raise RuntimeError(
        "Unexpected BrEaST external crop count: "
        f"{len(external_rows)} observed versus "
        f"{EXPECTED_BREAST_EXTERNAL_CROPS} expected."
    )

patient_sets = {
    split: set(
        development.loc[
            development["split"] == split,
            "global_patient_id",
        ]
    )
    for split in [
        "train",
        "validation",
        "test",
    ]
}

leakage = {
    "train_validation": len(
        patient_sets["train"]
        & patient_sets["validation"]
    ),
    "train_test": len(
        patient_sets["train"]
        & patient_sets["test"]
    ),
    "validation_test": len(
        patient_sets["validation"]
        & patient_sets["test"]
    ),
}

if any(leakage.values()):
    raise RuntimeError(
        f"Patient leakage detected: {leakage}"
    )

print(
    development
    .groupby("split")
    .size()
    .rename("crops")
)
print(
    development
    .groupby("split")["patient_id"]
    .nunique()
    .rename("patients")
)
print(
    "BrEaST rows kept sealed:",
    len(external_rows),
)
print(
    "Leakage audit:",
    leakage,
)

npz_search_roots = [
    DATA_ROOT,
    DATA_ROOT.parent,
]

if IN_KAGGLE:
    npz_search_roots.extend(
        [
            Path("/kaggle/input"),
            AUTO_EXTRACT_ROOT,
        ]
    )

if IN_COLAB:
    npz_search_roots.extend(
        [
            Path(
                "/content/drive/MyDrive/"
                "TRACKC_ULTRASOUND"
            ),
            AUTO_EXTRACT_ROOT,
        ]
    )

unique_search_roots = []
seen_roots = set()

for root in npz_search_roots:
    if not root.exists():
        continue

    try:
        key = str(root.resolve())
    except Exception:
        key = str(root)

    if key not in seen_roots:
        seen_roots.add(key)
        unique_search_roots.append(root)

all_npz_files = []
seen_npz = set()

for root in unique_search_roots:
    try:
        for path in root.rglob("*.npz"):
            try:
                key = str(path.resolve())
            except Exception:
                key = str(path)

            if key not in seen_npz:
                seen_npz.add(key)
                all_npz_files.append(path)
    except Exception:
        pass

print(
    "NPZ files indexed across candidate roots:",
    len(all_npz_files),
)

if len(all_npz_files) < EXPECTED_BUSBRA_CROPS:
    raise FileNotFoundError(
        "The manifest was loaded, but the full NPZ dataset "
        f"was not available. Only {len(all_npz_files)} NPZ files "
        f"were indexed; at least {EXPECTED_BUSBRA_CROPS} "
        "BUS-BRA NPZ files are required. Add the full dataset "
        "to Kaggle, not only ROI_US_Crops_256_v1_AUDIT_LIGHT."
    )

by_name = defaultdict(list)
by_npz_suffix = defaultdict(list)
by_dataset_split_sample = defaultdict(list)

for path in all_npz_files:
    by_name[path.name].append(path)

    normalized = str(path).replace("\\", "/")

    if "/npz/" in normalized:
        suffix = (
            "npz/"
            + normalized.split(
                "/npz/",
                1,
            )[1]
        )
        by_npz_suffix[suffix].append(path)

        suffix_parts = Path(suffix).parts

        if len(suffix_parts) >= 4:
            dataset_name = suffix_parts[-3]
            split_name = suffix_parts[-2]
            sample_name = Path(
                suffix_parts[-1]
            ).stem

            by_dataset_split_sample[
                (
                    dataset_name.lower(),
                    split_name.lower(),
                    sample_name,
                )
            ].append(path)

def _return_unique(
    candidates: Iterable[Path],
    method: str,
    row: pd.Series,
) -> Optional[Path]:
    existing = []

    for candidate in candidates:
        candidate = Path(candidate)

        if candidate.exists():
            existing.append(candidate)

    unique = []
    seen = set()

    for path in existing:
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)

        if key not in seen:
            seen.add(key)
            unique.append(path)

    if len(unique) == 1:
        return unique[0]

    if len(unique) > 1:
        raise RuntimeError(
            "Ambiguous NPZ resolution for "
            f"sample_id={row['sample_id']} using "
            f"method={method}: {unique[:10]}"
        )

    return None

def resolve_npz_row(
    row: pd.Series,
) -> Path:
    stored_text = str(
        row["npz_path"]
    ).replace("\\", "/")
    stored_path = Path(stored_text)
    dataset_name = str(
        row["dataset"]
    ).lower()
    split_name = str(
        row["split"]
    ).lower()
    sample_id = str(
        row["sample_id"]
    )
    stored_name = stored_path.name
    expected_name = (
        sample_id
        if sample_id.lower().endswith(".npz")
        else f"{sample_id}.npz"
    )

    resolved = _return_unique(
        [stored_path],
        "original_absolute_path",
        row,
    )
    if resolved is not None:
        return resolved

    deterministic_candidates = []

    for root in unique_search_roots:
        deterministic_candidates.extend(
            [
                root
                / "npz"
                / dataset_name
                / split_name
                / expected_name,
                root
                / "npz"
                / dataset_name
                / split_name
                / stored_name,
                root
                / "ROI_US_Crops_256_v1"
                / "npz"
                / dataset_name
                / split_name
                / expected_name,
                root
                / "ROI_US_Crops_256_v1"
                / "npz"
                / dataset_name
                / split_name
                / stored_name,
            ]
        )

    resolved = _return_unique(
        deterministic_candidates,
        "dataset_split_sample_path",
        row,
    )
    if resolved is not None:
        return resolved

    key_matches = (
        by_dataset_split_sample.get(
            (
                dataset_name,
                split_name,
                Path(expected_name).stem,
            ),
            [],
        )
        + by_dataset_split_sample.get(
            (
                dataset_name,
                split_name,
                Path(stored_name).stem,
            ),
            [],
        )
    )

    resolved = _return_unique(
        key_matches,
        "dataset_split_sample_index",
        row,
    )
    if resolved is not None:
        return resolved

    suffix_candidates = []

    if "/npz/" in stored_text:
        stored_suffix = (
            "npz/"
            + stored_text.split(
                "/npz/",
                1,
            )[1]
        )
        suffix_candidates.extend(
            by_npz_suffix.get(
                stored_suffix,
                [],
            )
        )

    expected_suffix = (
        f"npz/{dataset_name}/"
        f"{split_name}/{expected_name}"
    )
    suffix_candidates.extend(
        by_npz_suffix.get(
            expected_suffix,
            [],
        )
    )

    resolved = _return_unique(
        suffix_candidates,
        "relative_npz_suffix",
        row,
    )
    if resolved is not None:
        return resolved

    filename_matches = (
        by_name.get(expected_name, [])
        + by_name.get(stored_name, [])
    )

    resolved = _return_unique(
        filename_matches,
        "unique_filename",
        row,
    )
    if resolved is not None:
        return resolved

    raise FileNotFoundError(
        "Could not resolve the NPZ file for "
        f"sample_id={sample_id}, "
        f"dataset={dataset_name}, "
        f"split={split_name}, "
        f"stored_path={stored_text}. "
        "The most likely cause is that only the light audit "
        "package was added to Kaggle. Add "
        "ROI_US_Crops_256_v1_FULL.zip or the full extracted "
        "ROI_US_Crops_256_v1 directory."
    )

development[
    "resolved_npz_path"
] = [
    str(
        resolve_npz_row(row)
    )
    for _, row in tqdm(
        development.iterrows(),
        total=len(development),
        desc="Resolving development NPZ paths",
    )
]

if development[
    "resolved_npz_path"
].duplicated().any():
    duplicate_rows = development[
        development[
            "resolved_npz_path"
        ].duplicated(
            keep=False
        )
    ][
        [
            "sample_id",
            "dataset",
            "split",
            "resolved_npz_path",
        ]
    ]

    display(duplicate_rows.head(30))

    raise RuntimeError(
        "Duplicate resolved NPZ paths detected."
    )

missing_after_resolution = [
    path
    for path in development[
        "resolved_npz_path"
    ]
    if not Path(path).exists()
]

if missing_after_resolution:
    raise FileNotFoundError(
        "Some resolved NPZ paths do not exist: "
        f"{missing_after_resolution[:10]}"
    )

print(
    "Resolved BUS-BRA development NPZ files:",
    len(development),
)
print(
    "Path resolution audit: PASS"
)

In [ ]:
# Dataset and moderate ultrasound-specific augmentations
def random_affine_pair(
    image: np.ndarray,
    mask: np.ndarray,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    h, w = image.shape

    angle = float(rng.uniform(-10.0, 10.0))
    scale = float(rng.uniform(0.92, 1.08))
    tx = float(rng.uniform(-0.05, 0.05) * w)
    ty = float(rng.uniform(-0.05, 0.05) * h)

    matrix = cv2.getRotationMatrix2D(
        (w / 2.0, h / 2.0),
        angle,
        scale,
    )
    matrix[:, 2] += [tx, ty]

    image_warped = cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    mask_warped = cv2.warpAffine(
        mask,
        matrix,
        (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    return image_warped, (mask_warped > 0).astype(np.uint8)

def augment_ultrasound(
    image: np.ndarray,
    mask: np.ndarray,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    if rng.random() < 0.5:
        image = np.ascontiguousarray(image[:, ::-1])
        mask = np.ascontiguousarray(mask[:, ::-1])

    if rng.random() < 0.7:
        image, mask = random_affine_pair(
            image,
            mask,
            rng,
        )

    if rng.random() < 0.35:
        gamma = float(rng.uniform(0.85, 1.15))
        image = np.power(
            np.clip(image, 0, 1),
            gamma,
        ).astype(np.float32)

    if rng.random() < 0.25:
        noise = rng.normal(
            0.0,
            0.025,
            size=image.shape,
        ).astype(np.float32)
        image = np.clip(
            image + image * noise,
            0,
            1,
        )

    return image.astype(np.float32), mask.astype(np.uint8)

class UltrasoundROIDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        training: bool,
        seed: int,
    ):
        self.frame = frame.reset_index(drop=True).copy()
        self.training = training
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]

        with np.load(row["resolved_npz_path"]) as data:
            image = data["image"].astype(np.float32)
            mask = data["mask"].astype(np.uint8)

        if image.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(
                f"Unexpected image shape: {image.shape}"
            )
        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(
                f"Unexpected mask shape: {mask.shape}"
            )

        if self.training:
            rng = np.random.default_rng(
                self.seed
                + self.epoch * 1_000_003
                + index
            )
            image, mask = augment_ultrasound(
                image,
                mask,
                rng,
            )

        image_tensor = torch.from_numpy(
            np.stack([image, image, image], axis=0)
        ).float()
        mask_tensor = torch.from_numpy(
            mask[None, ...]
        ).float()

        metadata = {
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "global_patient_id": str(
                row["global_patient_id"]
            ),
            "case_id": str(row["case_id"]),
            "split": str(row["split"]),
            "pathology": str(
                row.get("pathology", "")
            ),
            "scanner": str(
                row.get("scanner", "")
            ),
            "crop_touches_border": int(
                row.get("crop_touches_border", 0)
            ),
            "lesion_ratio_crop": float(
                row.get("lesion_ratio_crop", np.nan)
            ),
        }

        return image_tensor, mask_tensor, metadata

def worker_init_fn(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

In [ ]:
try:
    from torchvision.models import swin_t
    TORCHVISION_OK = True
except Exception as error:
    swin_t = None
    TORCHVISION_OK = False
    print("torchvision Swin import error:", error)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(center), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.a4 = AttentionGate(base * 8, base * 8, base * 4)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.a3 = AttentionGate(base * 4, base * 4, base * 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.a2 = AttentionGate(base * 2, base * 2, base)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.a1 = AttentionGate(base, base, base // 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))
        u4 = self.u4(center)
        d4 = self.d4(
            torch.cat([u4, self.a4(u4, e4)], 1)
        )
        u3 = self.u3(d4)
        d3 = self.d3(
            torch.cat([u3, self.a3(u3, e3)], 1)
        )
        u2 = self.u2(d3)
        d2 = self.d2(
            torch.cat([u2, self.a2(u2, e2)], 1)
        )
        u1 = self.u1(d2)
        d1 = self.d1(
            torch.cat([u1, self.a1(u1, e1)], 1)
        )
        return self.out(d1)

class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_OK or swin_t is None:
            raise RuntimeError(
                "torchvision.models.swin_t unavailable"
            )
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.up3 = nn.ConvTranspose2d(512, 384, 2, 2)
        self.dec3 = ConvBlock(384 + 384, 256)
        self.up2 = nn.ConvTranspose2d(256, 192, 2, 2)
        self.dec2 = ConvBlock(192 + 192, 128)
        self.up1 = nn.ConvTranspose2d(128, 96, 2, 2)
        self.dec1 = ConvBlock(96 + 96, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec0 = ConvBlock(32, 32)
        self.up_final = nn.ConvTranspose2d(32, 32, 2, 2)
        self.out = nn.Conv2d(32, out_ch, 1)

    def _to_nchw(self, x):
        if (
            x.ndim == 4
            and x.shape[1] not in [96, 192, 384, 768]
        ):
            return x.permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        feats = []
        y = x
        for index, layer in enumerate(self.features):
            y = layer(y)
            if index in [1, 3, 5, 7]:
                feats.append(self._to_nchw(y))

        if len(feats) != 4:
            raise RuntimeError(
                f"Expected 4 Swin features, got {len(feats)}"
            )

        f1, f2, f3, f4 = feats
        center = self.center(f4)
        d3 = self.dec3(
            torch.cat([self.up3(center), f3], 1)
        )
        d2 = self.dec2(
            torch.cat([self.up2(d3), f2], 1)
        )
        d1 = self.dec1(
            torch.cat([self.up1(d2), f1], 1)
        )
        d0 = self.dec0(self.up0(d1))
        output = self.out(self.up_final(d0))

        if output.shape[-2:] != x.shape[-2:]:
            output = F.interpolate(
                output,
                size=x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return output

def build_model(name: str) -> nn.Module:
    if name == "unet":
        return UNet(in_ch=3, out_ch=1)
    if name == "attention_unet":
        return AttentionUNet(in_ch=3, out_ch=1)
    if name == "swin_tiny_unet":
        return SwinTinyUNet(out_ch=1)
    raise ValueError(name)

EXPECTED_PARAMS = {
    "unet": 7_763_041,
    "attention_unet": 7_851_773,
    "swin_tiny_unet": 38_350_819,
}

model = build_model(MODEL_NAME)
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model:", MODEL_NAME)
print("Parameters:", parameter_count)
assert parameter_count == EXPECTED_PARAMS[MODEL_NAME]

with torch.no_grad():
    probe = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    probe_output = model(probe)
assert probe_output.shape == (1, 1, IMAGE_SIZE, IMAGE_SIZE)

del probe, probe_output
gc.collect()
model = model.to(DEVICE)

In [ ]:
class DiceBCETverskyLoss(nn.Module):
    def __init__(
        self,
        bce_weight=1.0,
        dice_weight=1.0,
        tversky_weight=0.5,
        alpha=0.7,
        beta=0.3,
        gamma=0.75,
        smooth=1e-6,
    ):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.tversky_weight = tversky_weight
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
        )
        probs = torch.sigmoid(logits)

        dims = (1, 2, 3)
        intersection = (probs * targets).sum(dims)
        denominator = (
            probs.sum(dims)
            + targets.sum(dims)
        )
        dice = (
            2 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )
        dice_loss = 1 - dice.mean()

        tp = (probs * targets).sum(dims)
        fp = (probs * (1 - targets)).sum(dims)
        fn = ((1 - probs) * targets).sum(dims)
        tversky = (
            tp + self.smooth
        ) / (
            tp
            + self.alpha * fp
            + self.beta * fn
            + self.smooth
        )
        focal_tversky = (
            (1 - tversky) ** self.gamma
        ).mean()

        return (
            self.bce_weight * bce
            + self.dice_weight * dice_loss
            + self.tversky_weight * focal_tversky
        )

def soft_dice_batch(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * targets).sum(dims)
    denominator = probs.sum(dims) + targets.sum(dims)
    return (
        (2 * intersection + smooth)
        / (denominator + smooth)
    )

def binary_confusion(
    prediction: np.ndarray,
    target: np.ndarray,
) -> Tuple[int, int, int, int]:
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    tp = int(np.logical_and(prediction, target).sum())
    fp = int(
        np.logical_and(prediction, ~target).sum()
    )
    fn = int(
        np.logical_and(~prediction, target).sum()
    )
    tn = int(
        np.logical_and(~prediction, ~target).sum()
    )
    return tp, fp, fn, tn

def metrics_from_counts(
    tp: int,
    fp: int,
    fn: int,
    tn: int,
    smooth=1e-8,
) -> Dict[str, float]:
    return {
        "dice": (2 * tp + smooth)
        / (2 * tp + fp + fn + smooth),
        "iou": (tp + smooth)
        / (tp + fp + fn + smooth),
        "precision": (tp + smooth)
        / (tp + fp + smooth),
        "recall": (tp + smooth)
        / (tp + fn + smooth),
    }

def surface_distances(
    prediction: np.ndarray,
    target: np.ndarray,
) -> np.ndarray:
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    if not prediction.any() or not target.any():
        return np.array([], dtype=np.float32)

    prediction_surface = np.logical_xor(
        prediction,
        binary_erosion(prediction),
    )
    target_surface = np.logical_xor(
        target,
        binary_erosion(target),
    )

    distance_to_target = distance_transform_edt(
        ~target_surface
    )
    distance_to_prediction = distance_transform_edt(
        ~prediction_surface
    )

    distances = np.concatenate(
        [
            distance_to_target[prediction_surface],
            distance_to_prediction[target_surface],
        ]
    )
    return distances.astype(np.float32)

def hd95_asd(
    prediction: np.ndarray,
    target: np.ndarray,
) -> Tuple[float, float]:
    distances = surface_distances(
        prediction,
        target,
    )
    if distances.size == 0:
        return np.nan, np.nan
    return (
        float(np.percentile(distances, 95)),
        float(np.mean(distances)),
    )

criterion = DiceBCETverskyLoss()

In [ ]:
train_frame = development[
    development["split"] == "train"
].reset_index(drop=True)
validation_frame = development[
    development["split"] == "validation"
].reset_index(drop=True)
test_frame = development[
    development["split"] == "test"
].reset_index(drop=True)

batch_size = BATCH_SIZE_BY_MODEL[MODEL_NAME]

train_dataset = UltrasoundROIDataset(
    train_frame,
    training=True,
    seed=SEED,
)
validation_dataset = UltrasoundROIDataset(
    validation_frame,
    training=False,
    seed=SEED,
)
test_dataset = UltrasoundROIDataset(
    test_frame,
    training=False,
    seed=SEED,
)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False,
    worker_init_fn=worker_init_fn,
    generator=generator,
    persistent_workers=NUM_WORKERS > 0,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False,
    worker_init_fn=worker_init_fn,
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False,
    worker_init_fn=worker_init_fn,
    persistent_workers=NUM_WORKERS > 0,
)

print({
    "batch_size": batch_size,
    "train_batches": len(train_loader),
    "validation_batches": len(validation_loader),
    "test_batches": len(test_loader),
})

In [ ]:
def make_optimizer(model: nn.Module):
    if MODEL_NAME == "swin_tiny_unet":
        encoder_parameters = list(
            model.features.parameters()
        )
        encoder_ids = {
            id(parameter)
            for parameter in encoder_parameters
        }
        decoder_parameters = [
            parameter
            for parameter in model.parameters()
            if id(parameter) not in encoder_ids
        ]
        return torch.optim.AdamW(
            [
                {
                    "params": encoder_parameters,
                    "lr": SWIN_ENCODER_LR,
                },
                {
                    "params": decoder_parameters,
                    "lr": BASE_LR_BY_MODEL[MODEL_NAME],
                },
            ],
            weight_decay=WEIGHT_DECAY,
        )

    return torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR_BY_MODEL[MODEL_NAME],
        weight_decay=WEIGHT_DECAY,
    )

optimizer = make_optimizer(model)

def lr_lambda(epoch: int) -> float:
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(
            max(1, WARMUP_EPOCHS)
        )
    progress = (
        epoch - WARMUP_EPOCHS
    ) / max(
        1,
        MAX_EPOCHS - WARMUP_EPOCHS,
    )
    return 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda,
)

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP_ENABLED,
    )
except TypeError:
    scaler = torch.cuda.amp.GradScaler(
        enabled=AMP_ENABLED,
    )

def rng_state_dict() -> Dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state: Dict) -> None:
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if (
        torch.cuda.is_available()
        and "cuda" in state
    ):
        torch.cuda.set_rng_state_all(state["cuda"])

def save_checkpoint(
    path: Path,
    epoch: int,
    best_val_soft_dice: float,
    patience_counter: int,
    history: List[Dict],
) -> None:
    payload = {
        "model_name": MODEL_NAME,
        "seed": SEED,
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_soft_dice": best_val_soft_dice,
        "patience_counter": patience_counter,
        "history": history,
        "rng_state": rng_state_dict(),
        "parameter_count": parameter_count,
        "image_size": IMAGE_SIZE,
    }
    torch.save(payload, path)

def load_checkpoint(path: Path):
    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False,
    )
    if checkpoint.get("model_name") != MODEL_NAME:
        raise RuntimeError(
            "Checkpoint model_name mismatch."
        )
    if int(checkpoint.get("seed")) != int(SEED):
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )
    model.load_state_dict(
        checkpoint["model_state"],
        strict=True,
    )
    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )
    scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )
    scaler.load_state_dict(
        checkpoint.get("scaler_state", {})
    )
    restore_rng_state(
        checkpoint.get("rng_state", {})
    )
    return checkpoint

resume_path = None
if RESUME_CHECKPOINT_OVERRIDE:
    resume_path = Path(RESUME_CHECKPOINT_OVERRIDE)
elif LAST_PATH.exists():
    resume_path = LAST_PATH

start_epoch = 0
best_val_soft_dice = -np.inf
patience_counter = 0
history: List[Dict] = []

if resume_path is not None and resume_path.exists():
    checkpoint = load_checkpoint(resume_path)
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val_soft_dice = float(
        checkpoint["best_val_soft_dice"]
    )
    patience_counter = int(
        checkpoint["patience_counter"]
    )
    history = list(checkpoint.get("history", []))
    print(
        f"Resumed from {resume_path} at epoch {start_epoch}"
    )
else:
    print("Starting a new run.")

In [ ]:
def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        )
    return torch.autocast(
        device_type="cpu",
        enabled=False,
    )

def train_one_epoch(epoch: int) -> Dict[str, float]:
    model.train()
    train_dataset.set_epoch(epoch)

    losses = []
    soft_dices = []

    progress = tqdm(
        train_loader,
        desc=f"Train {epoch:03d}",
        leave=False,
    )

    for images, masks, metadata in progress:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        masks = masks.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_NORM,
        )
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            dice_values = soft_dice_batch(
                logits,
                masks,
            )

        losses.append(float(loss.detach().cpu()))
        soft_dices.extend(
            dice_values.detach().cpu().numpy().tolist()
        )

        progress.set_postfix(
            loss=f"{np.mean(losses):.4f}",
            dice=f"{np.mean(soft_dices):.4f}",
        )

    return {
        "train_loss": float(np.mean(losses)),
        "train_soft_dice": float(
            np.mean(soft_dices)
        ),
    }

@torch.inference_mode()
def validate_epoch() -> Dict[str, float]:
    model.eval()

    losses = []
    soft_dices = []

    for images, masks, metadata in tqdm(
        validation_loader,
        desc="Validation",
        leave=False,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        masks = masks.to(
            DEVICE,
            non_blocking=True,
        )

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        dice_values = soft_dice_batch(
            logits,
            masks,
        )

        losses.append(float(loss.detach().cpu()))
        soft_dices.extend(
            dice_values.detach().cpu().numpy().tolist()
        )

    return {
        "val_loss": float(np.mean(losses)),
        "val_soft_dice": float(
            np.mean(soft_dices)
        ),
    }

In [ ]:
config = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "parameter_count": parameter_count,
    "image_size": IMAGE_SIZE,
    "batch_size": batch_size,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "base_lr": BASE_LR_BY_MODEL[MODEL_NAME],
    "swin_encoder_lr": (
        SWIN_ENCODER_LR
        if MODEL_NAME == "swin_tiny_unet"
        else None
    ),
    "threshold_grid": THRESHOLD_GRID,
    "threshold_selection_level": "patient",
    "external_evaluation": False,
    "data_root": str(DATA_ROOT),
    "run_dir": str(RUN_DIR),
}
CONFIG_PATH.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

training_started = time.time()

for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_started = time.time()

    train_metrics = train_one_epoch(epoch)
    validation_metrics = validate_epoch()

    scheduler.step()

    row = {
        "epoch": epoch,
        **train_metrics,
        **validation_metrics,
        "lr_group_0": optimizer.param_groups[0]["lr"],
        "lr_group_1": (
            optimizer.param_groups[1]["lr"]
            if len(optimizer.param_groups) > 1
            else np.nan
        ),
        "epoch_seconds": time.time() - epoch_started,
    }
    history.append(row)

    improved = (
        validation_metrics["val_soft_dice"]
        > best_val_soft_dice + 1e-6
    )

    if improved:
        best_val_soft_dice = (
            validation_metrics["val_soft_dice"]
        )
        patience_counter = 0
        save_checkpoint(
            BEST_PATH,
            epoch,
            best_val_soft_dice,
            patience_counter,
            history,
        )
    else:
        patience_counter += 1

    save_checkpoint(
        LAST_PATH,
        epoch,
        best_val_soft_dice,
        patience_counter,
        history,
    )

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['train_loss']:.4f} | "
        f"val_loss={validation_metrics['val_loss']:.4f} | "
        f"val_soft_dice={validation_metrics['val_soft_dice']:.4f} | "
        f"best={best_val_soft_dice:.4f} | "
        f"patience={patience_counter}/"
        f"{EARLY_STOPPING_PATIENCE}"
    )

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print("Early stopping.")
        break

print(
    "Training duration (min):",
    (time.time() - training_started) / 60,
)
print("Best checkpoint:", BEST_PATH)

In [ ]:
@torch.inference_mode()
def collect_probabilities(
    loader: DataLoader,
    split_name: str,
) -> Tuple[pd.DataFrame, List[np.ndarray], List[np.ndarray]]:
    model.eval()

    metadata_rows = []
    probabilities = []
    targets = []

    for images, masks, metadata in tqdm(
        loader,
        desc=f"Inference {split_name}",
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with autocast_context():
            logits = model(images)

        batch_probabilities = (
            torch.sigmoid(logits)
            .detach()
            .cpu()
            .numpy()[:, 0]
        )
        batch_targets = (
            masks.detach().cpu().numpy()[:, 0]
        ).astype(np.uint8)

        batch_size_local = len(batch_probabilities)

        for index in range(batch_size_local):
            row = {
                key: (
                    value[index]
                    if isinstance(value, (list, tuple))
                    else (
                        value[index].item()
                        if torch.is_tensor(value)
                        else value
                    )
                )
                for key, value in metadata.items()
            }
            metadata_rows.append(row)
            probabilities.append(
                batch_probabilities[index]
                .astype(np.float32)
            )
            targets.append(
                batch_targets[index]
                .astype(np.uint8)
            )

    return (
        pd.DataFrame(metadata_rows),
        probabilities,
        targets,
    )

def evaluate_at_threshold(
    metadata_frame: pd.DataFrame,
    probabilities: List[np.ndarray],
    targets: List[np.ndarray],
    threshold: float,
    include_surface: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    crop_rows = []

    for index, (
        probability,
        target,
    ) in enumerate(zip(probabilities, targets)):
        prediction = (
            probability >= threshold
        ).astype(np.uint8)

        tp, fp, fn, tn = binary_confusion(
            prediction,
            target,
        )
        metrics = metrics_from_counts(
            tp,
            fp,
            fn,
            tn,
        )

        hd95, asd = (
            hd95_asd(prediction, target)
            if include_surface
            else (np.nan, np.nan)
        )

        row = metadata_frame.iloc[index].to_dict()
        row.update(
            {
                "threshold": threshold,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                **metrics,
                "hd95_px": hd95,
                "asd_px": asd,
                "prediction_empty": int(
                    prediction.sum() == 0
                ),
                "target_pixels": int(
                    target.sum()
                ),
                "prediction_pixels": int(
                    prediction.sum()
                ),
            }
        )
        crop_rows.append(row)

    crop_frame = pd.DataFrame(crop_rows)

    patient_rows = []
    for patient_id, group in crop_frame.groupby(
        "global_patient_id",
        sort=False,
    ):
        tp = int(group["tp"].sum())
        fp = int(group["fp"].sum())
        fn = int(group["fn"].sum())
        tn = int(group["tn"].sum())

        metrics = metrics_from_counts(
            tp,
            fp,
            fn,
            tn,
        )

        patient_rows.append(
            {
                "global_patient_id": patient_id,
                "patient_id": str(
                    group["patient_id"].iloc[0]
                ),
                "split": str(
                    group["split"].iloc[0]
                ),
                "pathology": str(
                    group["pathology"].iloc[0]
                ),
                "scanner": str(
                    group["scanner"].iloc[0]
                ),
                "n_crops": int(len(group)),
                "crop_touches_border_any": int(
                    group["crop_touches_border"].max()
                ),
                "lesion_ratio_crop_mean": float(
                    group["lesion_ratio_crop"].mean()
                ),
                "threshold": threshold,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                **metrics,
            }
        )

    patient_frame = pd.DataFrame(patient_rows)

    summary = {
        "threshold": float(threshold),
        "n_crops": int(len(crop_frame)),
        "n_patients": int(len(patient_frame)),
        "crop_dice_mean": float(
            crop_frame["dice"].mean()
        ),
        "crop_iou_mean": float(
            crop_frame["iou"].mean()
        ),
        "crop_precision_mean": float(
            crop_frame["precision"].mean()
        ),
        "crop_recall_mean": float(
            crop_frame["recall"].mean()
        ),
        "patient_dice_mean": float(
            patient_frame["dice"].mean()
        ),
        "patient_iou_mean": float(
            patient_frame["iou"].mean()
        ),
        "patient_precision_mean": float(
            patient_frame["precision"].mean()
        ),
        "patient_recall_mean": float(
            patient_frame["recall"].mean()
        ),
        "prediction_empty_count": int(
            crop_frame["prediction_empty"].sum()
        ),
    }

    if include_surface:
        summary["crop_hd95_mean_px"] = float(
            crop_frame["hd95_px"].mean()
        )
        summary["crop_asd_mean_px"] = float(
            crop_frame["asd_px"].mean()
        )

    return crop_frame, patient_frame, summary

In [ ]:
# Keep threshold selection restricted to validation data.
best_checkpoint = torch.load(
    BEST_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(
    best_checkpoint["model_state"],
    strict=True,
)

validation_metadata, validation_probs, validation_targets = (
    collect_probabilities(
        validation_loader,
        "validation",
    )
)

threshold_rows = []

for threshold in THRESHOLD_GRID:
    _, patient_frame, summary = evaluate_at_threshold(
        validation_metadata,
        validation_probs,
        validation_targets,
        threshold,
        include_surface=False,
    )
    threshold_rows.append(summary)

threshold_search = pd.DataFrame(
    threshold_rows
).sort_values(
    [
        "patient_dice_mean",
        "patient_iou_mean",
        "patient_recall_mean",
        "threshold",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)

selected_threshold = float(
    threshold_search.iloc[0]["threshold"]
)

threshold_search.to_csv(
    RUN_DIR / "validation_threshold_search.csv",
    index=False,
)

validation_crop, validation_patient, validation_summary = (
    evaluate_at_threshold(
        validation_metadata,
        validation_probs,
        validation_targets,
        selected_threshold,
        include_surface=True,
    )
)

test_metadata, test_probs, test_targets = (
    collect_probabilities(
        test_loader,
        "test",
    )
)

test_crop, test_patient, test_summary = (
    evaluate_at_threshold(
        test_metadata,
        test_probs,
        test_targets,
        selected_threshold,
        include_surface=True,
    )
)

validation_crop.to_csv(
    RUN_DIR / "validation_crop_metrics.csv",
    index=False,
)
validation_patient.to_csv(
    RUN_DIR / "validation_patient_metrics.csv",
    index=False,
)
test_crop.to_csv(
    RUN_DIR / "test_crop_metrics.csv",
    index=False,
)
test_patient.to_csv(
    RUN_DIR / "test_patient_metrics.csv",
    index=False,
)

summary = {
    "status": "completed_internal_evaluation",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "parameter_count": parameter_count,
    "best_epoch": int(
        best_checkpoint["epoch"]
    ),
    "best_val_soft_dice": float(
        best_checkpoint["best_val_soft_dice"]
    ),
    "selected_threshold": selected_threshold,
    "threshold_selection_split": "validation",
    "threshold_selection_level": "patient",
    "validation": validation_summary,
    "test": test_summary,
    "external_evaluated": False,
    "external_dataset": "BrEaST sealed",
}

(RUN_DIR / "run_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))

In [ ]:
readme = f"""
# Track C Phase 2 — {MODEL_NAME}, seed {SEED}

- Status: completed_internal_evaluation
- Architecture: {MODEL_NAME}
- Seed: {SEED}
- Parameters: {parameter_count}
- Selected threshold: {selected_threshold}
- Threshold source: BUS-BRA validation, patient-level
- Internal test: BUS-BRA test
- External BrEaST: NOT evaluated
- Best checkpoint: best.pt
- Resume checkpoint: last.pt
- Runtime: {RUNTIME_NAME}
"""

(RUN_DIR / "README.md").write_text(
    readme.strip(),
    encoding="utf-8",
)

LIGHT_ZIP_PATH = (
    OUTPUT_ROOT
    / f"{RUN_ID}_RESULTS_LIGHT.zip"
)
FULL_ZIP_PATH = (
    OUTPUT_ROOT
    / f"{RUN_ID}_RESULTS_FULL.zip"
)

if CREATE_LIGHT_ZIP:
    excluded_names = {
        "best.pt",
        "last.pt",
    }

    with zipfile.ZipFile(
        LIGHT_ZIP_PATH,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in RUN_DIR.rglob("*"):
            if (
                path.is_file()
                and path.name not in excluded_names
            ):
                archive.write(
                    path,
                    path.relative_to(OUTPUT_ROOT),
                )

with zipfile.ZipFile(
    FULL_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as archive:
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(OUTPUT_ROOT),
            )

PUBLISHED_FILES = {}

if IN_KAGGLE:
    publish_root = Path(
        "/kaggle/working"
    )

    top_level_light = (
        publish_root
        / LIGHT_ZIP_PATH.name
    )
    top_level_full = (
        publish_root
        / FULL_ZIP_PATH.name
    )
    top_level_best = (
        publish_root
        / f"{RUN_ID}_best.pt"
    )
    top_level_last = (
        publish_root
        / f"{RUN_ID}_last.pt"
    )
    top_level_summary = (
        publish_root
        / f"{RUN_ID}_run_summary.json"
    )

    if CREATE_LIGHT_ZIP:
        shutil.copy2(
            LIGHT_ZIP_PATH,
            top_level_light,
        )
        PUBLISHED_FILES[
            "light_results_zip"
        ] = str(top_level_light)

    shutil.copy2(
        FULL_ZIP_PATH,
        top_level_full,
    )
    shutil.copy2(
        BEST_PATH,
        top_level_best,
    )
    shutil.copy2(
        LAST_PATH,
        top_level_last,
    )
    shutil.copy2(
        RUN_DIR / "run_summary.json",
        top_level_summary,
    )

    PUBLISHED_FILES.update(
        {
            "full_results_zip": str(
                top_level_full
            ),
            "best_checkpoint": str(
                top_level_best
            ),
            "last_checkpoint": str(
                top_level_last
            ),
            "run_summary": str(
                top_level_summary
            ),
        }
    )

else:
    PUBLISHED_FILES = {
        "light_results_zip": (
            str(LIGHT_ZIP_PATH)
            if CREATE_LIGHT_ZIP
            else None
        ),
        "full_results_zip": str(
            FULL_ZIP_PATH
        ),
        "best_checkpoint": str(
            BEST_PATH
        ),
        "last_checkpoint": str(
            LAST_PATH
        ),
        "run_summary": str(
            RUN_DIR / "run_summary.json"
        ),
    }

publication_manifest = {
    "runtime": RUNTIME_NAME,
    "model_name": MODEL_NAME,
    "seed": SEED,
    "run_directory": str(RUN_DIR),
    "published_files": PUBLISHED_FILES,
}

publication_manifest_path = (
    Path("/kaggle/working")
    / f"{RUN_ID}_OUTPUT_LOCATIONS.json"
    if IN_KAGGLE
    else RUN_DIR
    / "OUTPUT_LOCATIONS.json"
)

publication_manifest_path.write_text(
    json.dumps(
        publication_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nOUTPUT PUBLICATION SUMMARY"
)
print(
    json.dumps(
        publication_manifest,
        indent=2,
    )
)

if IN_KAGGLE:
    print(
        "\nKaggle Output tab files:"
    )
    for key, value in PUBLISHED_FILES.items():
        if value:
            path = Path(value)
            print(
                f"- {key}: {path} "
                f"({path.stat().st_size / (1024 ** 2):.2f} MB)"
            )

print(
    "\nBrEaST remains sealed. "
    "Do not evaluate it before all nine runs are complete."
)

In [ ]:

legacy_root = Path(
    "/content/drive/MyDrive/"
    "TRACKC_ULTRASOUND/"
    "TRACKC_PHASE2_RESULTS"
)
kaggle_recovery_root = Path(
    "/kaggle/working/"
    "TRACKC_PHASE2_RESULTS_RECOVERED"
)

if IN_KAGGLE and legacy_root.exists():
    kaggle_recovery_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    legacy_run_dir = (
        legacy_root / RUN_ID
    )
    legacy_light_zip = (
        legacy_root
        / f"{RUN_ID}_RESULTS_LIGHT.zip"
    )

    if legacy_run_dir.exists():
        recovered_run_dir = (
            kaggle_recovery_root / RUN_ID
        )
        shutil.copytree(
            legacy_run_dir,
            recovered_run_dir,
            dirs_exist_ok=True,
        )

        recovered_full_zip = (
            Path("/kaggle/working")
            / f"{RUN_ID}_RECOVERED_FULL.zip"
        )

        with zipfile.ZipFile(
            recovered_full_zip,
            "w",
            compression=zipfile.ZIP_DEFLATED,
            allowZip64=True,
        ) as archive:
            for path in recovered_run_dir.rglob("*"):
                if path.is_file():
                    archive.write(
                        path,
                        path.relative_to(
                            kaggle_recovery_root
                        ),
                    )

        print(
            "Recovered full ZIP:",
            recovered_full_zip,
        )

    if legacy_light_zip.exists():
        recovered_light_zip = (
            Path("/kaggle/working")
            / f"{RUN_ID}_RECOVERED_LIGHT.zip"
        )
        shutil.copy2(
            legacy_light_zip,
            recovered_light_zip,
        )
        print(
            "Recovered light ZIP:",
            recovered_light_zip,
        )

    if not legacy_run_dir.exists():
        print(
            "No active legacy run directory was found at:",
            legacy_run_dir,
        )
else:
    print(
        "Legacy recovery is unavailable. "
        "The previous runtime may have ended, or the legacy path "
        "does not exist in this session."
    )